# RodentFlow · 把现有模型导出成 ONNX（Windows 版用）

从 Drive 上各模型的 `best.pt` 直接导出 ONNX（FP32，固定输入尺寸），不重新训练。

每个模型做四件事：
1. 找到 checkpoint，算 SHA-256，和 RodentFlow 发布模型的前缀对照；
2. 导出 ONNX，输出布局和 App 里的 Core ML 包相同（6 点模型 `[1,300,24]`，圆筒模型 `[1,26,8400]`）；
3. 同一个输入张量分别过 PyTorch 和 ONNX Runtime，比较输出差；
4. 写一份 `<模型名>.json`（输入输出、关键点名、哈希、版本、比对结果）。

结果在 `MyDrive/rodentflow_windows/onnx_<运行时间>/`，另打一个 zip。

- 不需要 GPU。全部运行即可；断线后再全部运行会跳过已完成的模型。
- 不会往各模型的 `runs/` 文件夹里写东西（先把 `.pt` 复制到 Colab 本地再导出）。
- **不扫描整个 Drive。** 在第 2 格的 `MODELS` 里填每个 checkpoint 的路径；路径不对时，只在你给的 `search` 文件夹里找。
- 没填路径的模型会跳过。
- **MouseDetector 不在这里**：它没有 `.pt`，只有公开仓库里的 `.mlpackage`，要另想办法。

In [ ]:
# 1. 安装（版本固定为多鼠模型训练时用的 8.4.208）
!pip install -q "ultralytics==8.4.208" onnx onnxruntime onnxslim

In [ ]:
# 2. 设置
import os, sys, json, time, hashlib, shutil, glob, csv, datetime, platform

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
    WORK = '/content/rf_onnx_work'
except ImportError:                       # 不在 Colab：用环境变量指定 Drive 根目录
    DRIVE = os.environ.get('RF_DRIVE', os.path.expanduser('~/drive'))
    WORK = os.environ.get('RF_WORK', os.path.abspath('rf_onnx_work'))

RUN = os.environ.get('RF_RUN') or datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_ROOT = os.path.join(DRIVE, 'rodentflow_windows')
# 断线后接着跑：如果有没写完 MANIFEST.json 的上一次运行，就接着用那个文件夹
_unfinished = [d for d in sorted(glob.glob(os.path.join(OUT_ROOT, 'onnx_*')))
               if os.path.isdir(d) and not os.path.exists(os.path.join(d, 'MANIFEST.json'))]
OUT = _unfinished[-1] if _unfinished and not os.environ.get('RF_RUN') else os.path.join(OUT_ROOT, f'onnx_{RUN}')
os.makedirs(OUT, exist_ok=True); os.makedirs(WORK, exist_ok=True)

OPSET = 17            # DirectML 支持到 opset 20；17 各家运行时都能读
FORCE = False         # True = 已导出的也重做
SEARCH_IF_MISSING = True   # 路径不存在时，只在该模型 search 里列出的文件夹内找 .pt（不会扫整个 Drive）
PARITY_TOL_PX = 0.05  # PyTorch 和 ONNX 的坐标差超过这个值就标出来（模型输入像素）
PARITY_TOL_CONF = 0.002

# 可选：给比对用的真实画面（视频或图片文件夹，路径相对 MyDrive）。不给就用合成图。
# 例：TEST_SOURCE = {'*': 'my_videos/openfield.mp4', 'CylinderPose7': 'my_videos/cylinder.mp4'}
TEST_SOURCE = {}
N_TEST_FRAMES = 8
GRAYSCALE_TEST = True   # App 默认把画面转灰度再送进模型

# head：'e2e' = 端到端（NMS-free，[1,300,6+3K]）；'raw' = 原始输出（[1,5+3K,锚点数]，要自己做 NMS）
# sha = RodentFlow 发布模型的 checkpoint SHA-256 前缀（不核对就写 None）
# paths = 依次尝试的路径（相对 MyDrive）；search = 找不到时去找的文件夹（相对 MyDrive，只在这些文件夹里面找）
#
# 公开版：checkpoint 路径都是空的。把你自己的 best.pt 路径填进 paths（或把它所在的文件夹填进 search），
# 没填的模型会跳过。sha 是 RodentFlow 发布的模型对应的 checkpoint 的 SHA-256 前缀：
# 填了的话，notebook 会核对你的文件是不是同一个 checkpoint。
MODELS = [
    dict(name='MousePose6', imgsz=960, head='e2e', kpts=6, sha=None,
         paths=[],
         search=[], note='单鼠 n-960'),
    dict(name='MousePose6Dyad', imgsz=960, head='e2e', kpts=6, sha='01572089',
         paths=[],
         search=[], note='老双鼠 s-960'),
    dict(name='WhiteMousePose6', imgsz=960, head='e2e', kpts=6, sha='5399e7fd81e6',
         paths=[],
         search=[], note='白鼠 n-960'),
    dict(name='MousePose6DyadBW', imgsz=960, head='e2e', kpts=6, sha='1c5082e7',
         paths=[],
         search=[], note='黑白双鼠 s-960 v2'),
    dict(name='GroupPose6', imgsz=960, head='e2e', kpts=6, sha=None,
         paths=[],
         search=[], note='多鼠 s-960 v1（App 里输出后还要做 pose NMS）'),
    dict(name='CylinderPose7', imgsz=640, head='raw', kpts=7, sha=None,
         paths=[],
         search=[], note='圆筒肢体模型 A；Core ML 包输出是 [1,26,8400]'),
    dict(name='PalmReaderPose8', imgsz=640, head='raw', kpts=8, sha='88220bb1a9594ccb',
         paths=[],
         search=[],
         note='head 按圆筒模型假定为 raw，没有核对过'),
]
# 在 search 文件夹里找的时候跳过的子文件夹（数据集）
SKIP_DIRS = {'datasets', 'images', 'labels', 'rodentflow_windows', '.Trash', '__pycache__'}

print('Drive :', DRIVE)
print('输出  :', OUT)

In [ ]:
# 3. 找 checkpoint
def sha256(path, buf=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(buf)
            if not b: break
            h.update(b)
    return h.hexdigest()

def pt_files_under(rel_dirs):
    # 只在给定的文件夹里找 .pt（跳过数据集子文件夹）。不扫整个 Drive。
    found = []
    for rel in rel_dirs:
        top = os.path.join(DRIVE, rel)
        if not rel.strip('/') or not os.path.isdir(top):
            continue
        for root, dirs, files in os.walk(top):
            dirs[:] = [d for d in dirs if d not in SKIP_DIRS and not d.startswith('.')]
            found += [os.path.join(root, f) for f in files if f.endswith('.pt')]
    return found

def locate(spec):
    # 返回 (路径或 None, 状态, sha256 或 None, 说明)
    tried = []
    for rel in spec['paths']:
        p = os.path.join(DRIVE, rel)
        if os.path.isfile(p):
            h = sha256(p)
            if spec['sha'] is None:
                return p, 'found', h, '路径存在；没有给这个模型的 SHA-256 前缀，没有核对'
            if h.startswith(spec['sha']):
                return p, 'found_sha_ok', h, 'SHA-256 前缀和记录一致'
            tried.append(f'{rel}（存在，但 SHA-256 是 {h[:12]}…，记录是 {spec["sha"]}…）')
        else:
            tried.append(f'{rel}（不存在）')
    if SEARCH_IF_MISSING:
        if not spec['search']:
            return None, 'missing', None, '；'.join(tried + ['没有填地址。在第 2 格的 MODELS 里填 paths，不填就跳过'])
        cands = pt_files_under(spec['search'])
        if spec['sha']:
            for p in cands:
                h = sha256(p)
                if h.startswith(spec['sha']):
                    return p, 'found_by_search_sha_ok', h, '在 search 文件夹里找到，SHA-256 前缀和记录一致'
            return None, 'missing', None, '；'.join(tried) + f'；在 {spec["search"]} 里找到 {len(cands)} 个 .pt，SHA-256 都对不上'
        show = [os.path.relpath(p, DRIVE) for p in cands[:12]]
        return None, 'missing', None, '；'.join(tried) + '；没有 SHA-256 记录，不自动挑选。候选：' + ' | '.join(show)
    return None, 'missing', None, '；'.join(tried)

located = {}
for spec in MODELS:
    p, status, h, msg = locate(spec)
    located[spec['name']] = dict(path=p, status=status, sha256=h, msg=msg)
    print(f"{spec['name']:<18} {status:<24} {os.path.relpath(p, DRIVE) if p else '—'}")
    print(f"{'':<18} {msg}")
missing = [n for n, v in located.items() if v['path'] is None]
if missing:
    print('\n找不到：', ', '.join(missing), '→ 在第 2 格的 MODELS 里把 paths 改成正确路径后重新运行。其余模型照常导出。')

In [ ]:
# 4. 导出 + 比对用的函数
import numpy as np, torch, onnx, onnxruntime as ort, cv2, ultralytics
from ultralytics import YOLO

def expected_shape(head, k, imgsz, nc=1):
    if head == 'e2e':
        return [1, 300, 6 + 3 * k]
    return [1, 4 + nc + 3 * k, sum((imgsz // s) ** 2 for s in (8, 16, 32))]

def letterbox(img, size, gray=True):
    # BGR 图 → size×size，等比缩放，四周补 114。和 Ultralytics 的 LetterBox 相同的做法。
    if gray:
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY); img = cv2.merge([g, g, g])
    h, w = img.shape[:2]; r = min(size / h, size / w)
    nw, nh = int(round(w * r)), int(round(h * r))
    im = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((size, size, 3), 114, np.uint8)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = im
    return canvas

def to_tensor(canvas):
    return np.ascontiguousarray(canvas[:, :, ::-1].transpose(2, 0, 1)[None]).astype(np.float32) / 255.0

def test_inputs(name, size):
    # 返回 (输入张量列表, 来源说明)
    rel = TEST_SOURCE.get(name) or TEST_SOURCE.get('*')
    frames = []
    if rel:
        src = os.path.join(DRIVE, rel)
        if os.path.isdir(src):
            files = sorted(f for f in glob.glob(os.path.join(src, '*')) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
            for f in files[::max(1, len(files) // N_TEST_FRAMES)][:N_TEST_FRAMES]:
                im = cv2.imread(f)
                if im is not None: frames.append(im)
        elif os.path.isfile(src):
            cap = cv2.VideoCapture(src); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            for i in np.linspace(0, max(n - 1, 0), N_TEST_FRAMES).astype(int):
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, im = cap.read()
                if ok: frames.append(im)
            cap.release()
        if frames:
            return [to_tensor(letterbox(f, size, GRAYSCALE_TEST)) for f in frames], f'{len(frames)} 帧真实画面：{rel}'
        print(f'  ⚠ 读不到 TEST_SOURCE：{rel}，改用合成图')
    rng = np.random.default_rng(0); xs = []
    for i in range(4):   # 合成图：平滑亮度场 + 几个深色椭圆，只为了让网络各层都有非平凡的激活
        base = cv2.resize(rng.integers(60, 200, (12, 12)).astype(np.uint8), (size, size), interpolation=cv2.INTER_CUBIC)
        img = cv2.merge([base, base, base])
        for _ in range(3):
            c = tuple(int(v) for v in rng.integers(size // 6, size * 5 // 6, 2))
            cv2.ellipse(img, c, (size // 14, size // 32), float(rng.integers(0, 180)), 0, 360, (25, 25, 25), -1)
        xs.append(to_tensor(img))
    return xs, '4 张合成图（没有鼠；只说明两边算出的数相同，不说明检测效果）'

def torch_reference(pt, head, xs):
    # 用和导出相同的 head 跑 PyTorch。
    net = YOLO(pt).model.eval().float()
    net.end2end = (head == 'e2e')
    outs = []
    with torch.no_grad():
        for x in xs:
            y = net(torch.from_numpy(x))
            y = y[0] if isinstance(y, (list, tuple)) else y
            outs.append(y.cpu().numpy())
    return outs

def compare(head, k, ref, got, conf=0.25):
    # 返回 dict：比较 PyTorch(ref) 和 ONNX(got) 的输出
    r = dict(max_xy_diff_px=0.0, max_score_diff=0.0, max_kpt_conf_diff=0.0, n_det_torch=0, n_det_onnx=0, unmatched=0)
    for a, b in zip(ref, got):
        a, b = a[0], b[0]
        if head == 'e2e':            # 行 = [x1,y1,x2,y2,score,class,(x,y,conf)×K]；行的顺序可能因并列分数不同，按框匹配
            A, B = a[a[:, 4] > conf], b[b[:, 4] > conf]
            r['n_det_torch'] += len(A); r['n_det_onnx'] += len(B)
            for row in A:
                if not len(B): r['unmatched'] += 1; continue
                j = int(np.abs(B[:, :4] - row[:4]).sum(1).argmin()); q = B[j]
                if np.abs(q[:4] - row[:4]).max() > 2.0: r['unmatched'] += 1; continue
                kp, kq = row[6:].reshape(k, 3), q[6:].reshape(k, 3)
                r['max_xy_diff_px'] = max(r['max_xy_diff_px'], float(np.abs(q[:4] - row[:4]).max()), float(np.abs(kq[:, :2] - kp[:, :2]).max()))
                r['max_score_diff'] = max(r['max_score_diff'], float(abs(q[4] - row[4])))
                r['max_kpt_conf_diff'] = max(r['max_kpt_conf_diff'], float(np.abs(kq[:, 2] - kp[:, 2]).max()))
            # 分数 ≤ conf 的行也看一眼最高分，防止“两边都没有检测”时比对是空的
            r['max_score_diff'] = max(r['max_score_diff'], float(abs(a[:, 4].max() - b[:, 4].max())))
        else:                        # [4+nc+3K, 锚点]，锚点顺序固定，逐元素比较
            nc = a.shape[0] - 4 - 3 * k
            ka, kb = a[4 + nc:].reshape(k, 3, -1), b[4 + nc:].reshape(k, 3, -1)
            r['max_xy_diff_px'] = max(r['max_xy_diff_px'], float(np.abs(a[:4] - b[:4]).max()), float(np.abs(ka[:, :2] - kb[:, :2]).max()))
            r['max_score_diff'] = max(r['max_score_diff'], float(np.abs(a[4:4 + nc] - b[4:4 + nc]).max()))
            r['max_kpt_conf_diff'] = max(r['max_kpt_conf_diff'], float(np.abs(ka[:, 2] - kb[:, 2]).max()))
            r['n_det_torch'] += int((a[4:4 + nc].max(0) > conf).sum()); r['n_det_onnx'] += int((b[4:4 + nc].max(0) > conf).sum())
    r['pass'] = bool(r['max_xy_diff_px'] <= PARITY_TOL_PX and r['max_score_diff'] <= PARITY_TOL_CONF
                     and r['max_kpt_conf_diff'] <= PARITY_TOL_CONF and r['unmatched'] == 0
                     and r['n_det_torch'] == r['n_det_onnx'])
    return r

def export_one(spec, loc):
    name, size, head, k = spec['name'], spec['imgsz'], spec['head'], spec['kpts']
    onnx_name = f'{name}_{size}_fp32.onnx'
    out_onnx, out_json = os.path.join(OUT, onnx_name), os.path.join(OUT, f'{name}.json')
    if not FORCE and os.path.exists(out_onnx) and os.path.exists(out_json):
        info = json.load(open(out_json))
        if info.get('checkpoint_sha256') == loc['sha256'] and info.get('onnx_sha256') == sha256(out_onnx):
            print('  已导出过，跳过'); return info
    wd = os.path.join(WORK, name); shutil.rmtree(wd, ignore_errors=True); os.makedirs(wd)
    pt = os.path.join(wd, f'{name}.pt'); shutil.copy(loc['path'], pt)

    m = YOLO(pt)
    head_mod = m.model.model[-1]
    ck_k = list(getattr(head_mod, 'kpt_shape', [None, None]))
    if m.task != 'pose' or ck_k[0] != k:
        raise RuntimeError(f'checkpoint 是 task={m.task}、kpt_shape={ck_k}，设置里写的是 {k} 个点。检查路径是否指错了模型。')
    if head == 'e2e' and getattr(head_mod, 'one2one_cv2', None) is None:
        raise RuntimeError('这个 checkpoint 没有 one-to-one 分支，导不出端到端输出。把 head 改成 raw。')
    names = dict(m.names)
    kpt_names = getattr(m.model, 'kpt_names', None)
    train_args = getattr(m.model, 'args', None) or {}
    train_imgsz = train_args.get('imgsz') if isinstance(train_args, dict) else getattr(train_args, 'imgsz', None)

    kw = dict(nms=False) if head == 'e2e' else {}
    f = m.export(format='onnx', imgsz=size, opset=OPSET, simplify=True, dynamic=False, batch=1, device='cpu', **kw)

    mo = onnx.load(f); onnx.checker.check_model(mo)
    for key, val in {'rodentflow.model': name, 'rodentflow.checkpoint_sha256': loc['sha256'],
                     'rodentflow.head': head, 'rodentflow.export_run': os.path.basename(OUT)}.items():
        p = mo.metadata_props.add(); p.key, p.value = key, str(val)
    onnx.save(mo, f)
    opset = max(o.version for o in mo.opset_import if o.domain in ('', 'ai.onnx'))
    ops = sorted({n.op_type for n in mo.graph.node})

    sess = ort.InferenceSession(f, providers=['CPUExecutionProvider'])
    i0, o0 = sess.get_inputs()[0], sess.get_outputs()[0]
    exp = expected_shape(head, k, size, len(names))
    if list(o0.shape) != exp or list(i0.shape) != [1, 3, size, size] or len(sess.get_outputs()) != 1:
        raise RuntimeError(f'输出形状 {list(o0.shape)}（输入 {list(i0.shape)}，{len(sess.get_outputs())} 个输出），应为 {exp}')

    xs, src = test_inputs(name, size)
    got = [sess.run(None, {i0.name: x})[0] for x in xs]
    par = compare(head, k, torch_reference(pt, head, xs), got); par['inputs'] = src
    t0 = time.perf_counter()
    for _ in range(5): sess.run(None, {i0.name: xs[0]})
    cpu_ms = (time.perf_counter() - t0) / 5 * 1000

    shutil.copy(f, out_onnx)
    if head == 'e2e':
        layout = ('每行一个检测，共 300 行，按分数从高到低：[x1, y1, x2, y2, score, class, (x, y, conf) × K]。'
                  '坐标是模型输入（letterbox 后）的像素，框是 xyxy；不需要 NMS。')
    else:
        layout = ('[4 + nc + 3K, 锚点数]：每个锚点一列，[cx, cy, w, h, 各类分数, (x, y, conf) × K]。'
                  '坐标是模型输入（letterbox 后）的像素，框是 xywh；需要自己做 NMS。')
    info = dict(
        model=name, note=spec['note'], onnx_file=onnx_name, onnx_sha256=sha256(out_onnx),
        onnx_size_mb=round(os.path.getsize(out_onnx) / 1e6, 2), precision='fp32', opset=opset, op_types=ops,
        checkpoint_drive_path=os.path.relpath(loc['path'], DRIVE), checkpoint_sha256=loc['sha256'],
        checkpoint_recorded_sha_prefix=spec['sha'], checkpoint_status=loc['status'],
        input=dict(name=i0.name, shape=list(i0.shape), dtype='float32', color='RGB', range='0–1', layout='NCHW',
                   preprocessing=f'等比缩放到 {size}×{size} 以内，居中，四周补灰（Ultralytics 用 114/255）'),
        output=dict(name=o0.name, shape=list(o0.shape), dtype='float32', head=head, layout=layout),
        kpt_shape=ck_k, kpt_names=kpt_names, class_names=names, train_imgsz=train_imgsz,
        parity_vs_pytorch=par, onnxruntime_cpu_ms_per_frame=round(cpu_ms, 1), cpu=platform.processor() or platform.machine(),
        versions=dict(ultralytics=ultralytics.__version__, torch=torch.__version__, onnx=onnx.__version__,
                      onnxruntime=ort.__version__, python=platform.python_version()),
        exported_at=datetime.datetime.now().isoformat(timespec='seconds'))
    json.dump(info, open(out_json, 'w'), ensure_ascii=False, indent=2)
    return info
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| onnx', onnx.__version__, '| onnxruntime', ort.__version__)

In [ ]:
# 5. 逐个导出（一个失败不影响其他）
results = {}
for spec in MODELS:
    name, loc = spec['name'], located[spec['name']]
    print(f'\n===== {name} =====')
    if loc['path'] is None:
        results[name] = dict(model=name, status='missing', error=loc['msg']); print('  找不到 checkpoint，跳过'); continue
    try:
        info = export_one(spec, loc); info['status'] = 'ok'; results[name] = info
        p = info['parity_vs_pytorch']
        print(f"  → {info['onnx_file']}  {info['onnx_size_mb']} MB  输出 {info['output']['shape']}  opset {info['opset']}")
        print(f"  和 PyTorch 比：坐标最大差 {p['max_xy_diff_px']:.4f} px，分数最大差 {p['max_score_diff']:.5f}，"
              f"检测数 {p['n_det_torch']} / {p['n_det_onnx']} → {'一致' if p['pass'] else '⚠ 超出容差'}")
    except Exception as e:
        results[name] = dict(model=name, status='failed', error=f'{type(e).__name__}: {e}')
        print('  ✗ 失败：', results[name]['error'])

In [ ]:
# 6. 汇总、写清单、打包
rows = []
for spec in MODELS:
    r = results[spec['name']]; p = r.get('parity_vs_pytorch', {})
    rows.append(dict(
        model=spec['name'], status=r['status'], checkpoint=r.get('checkpoint_drive_path', ''),
        checkpoint_sha256=r.get('checkpoint_sha256', ''), sha_check=located[spec['name']]['status'],
        onnx=r.get('onnx_file', ''), size_mb=r.get('onnx_size_mb', ''), imgsz=spec['imgsz'], head=spec['head'],
        output_shape='×'.join(map(str, r['output']['shape'])) if 'output' in r else '',
        max_xy_diff_px=p.get('max_xy_diff_px', ''), max_score_diff=p.get('max_score_diff', ''),
        parity='一致' if p.get('pass') else ('超出容差' if p else ''), parity_inputs=p.get('inputs', ''),
        cpu_ms=r.get('onnxruntime_cpu_ms_per_frame', ''), error=r.get('error', '')))
with open(os.path.join(OUT, 'export_summary.csv'), 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

ok = [r for r in rows if r['status'] == 'ok']
not_ok = [r for r in rows if r['status'] != 'ok']
manifest = dict(run=os.path.basename(OUT), finished_at=datetime.datetime.now().isoformat(timespec='seconds'),
                complete=not not_ok, models=results,
                not_included=['MouseDetector：没有 .pt，只有公开仓库里的 TSTMulti/Models/MouseDetector.mlpackage'])
if not not_ok:   # 只有全部成功才写 MANIFEST.json；否则下次运行会接着用这个文件夹
    json.dump(manifest, open(os.path.join(OUT, 'MANIFEST.json'), 'w'), ensure_ascii=False, indent=2)
else:
    json.dump(manifest, open(os.path.join(OUT, 'MANIFEST_partial.json'), 'w'), ensure_ascii=False, indent=2)

zip_path = shutil.make_archive(os.path.join(OUT_ROOT, f'RodentFlow_ONNX_{os.path.basename(OUT)[5:]}'), 'zip', OUT)

print(f"{'模型':<18}{'状态':<9}{'输出':<14}{'MB':>6}  {'坐标差 px':>10}  {'比对':<8}{'CPU ms':>7}")
for r in rows:
    xy = f"{r['max_xy_diff_px']:.4f}" if r['max_xy_diff_px'] != '' else ''
    print(f"{r['model']:<18}{r['status']:<9}{r['output_shape']:<14}{str(r['size_mb']):>6}  {xy:>10}  {r['parity']:<8}{str(r['cpu_ms']):>7}")
print(f'\n成功 {len(ok)} / {len(rows)}')
for r in not_ok:
    print(f"  {r['model']}: {r['status']} — {r['error']}")
sha_unverified = [r['model'] for r in ok if r['sha_check'] == 'found']
if sha_unverified:
    print('没有 SHA-256 记录可对照的（只确认了路径存在）：', ', '.join(sha_unverified))
print('\n文件夹：', OUT)
print('zip   ：', zip_path, f'（{os.path.getsize(zip_path) / 1e6:.1f} MB）')
if not_ok:
    print('有模型没导出：改好第 2 格的路径后再全部运行，会接着用同一个文件夹，已完成的会跳过。')

## Windows 端怎么用这些文件

- 每个 `.onnx` 配一个同名 `.json`，里面写了输入输出名、形状、关键点名和布局。ONNX 文件自己的元数据里也有（Ultralytics 写的 `kpt_names`、`imgsz`、`end2end`，加上 `rodentflow.*`）。
- 输入：`float32`，`[1, 3, S, S]`，RGB，0–1，NCHW。画面等比缩放后居中，四周补灰。
- `e2e` 模型（6 点的五个）：输出 `[1, 300, 24]`，和 Core ML 包相同，`PoseDecoding` 的逻辑可以照搬。
- `raw` 模型（圆筒、PalmReader）：输出 `[1, 5 + 3K, 8400]`，要自己做 NMS。

## 这本 notebook 没有做的

- **没有和 Core ML 包比**（Colab 上跑不了 Core ML）。这里只证明 ONNX 和 PyTorch 一致；Core ML 的 INT8 / FP16 包本身和 PyTorch 有量化差，那一步要在 Mac 上比。
- **没有量化**。都是 FP32，文件比 App 里的包大（n 约 12 MB，s 约 40 MB）。
- **没有在 GPU 后端上跑**（DirectML、OpenVINO、TensorRT）。CPU ms/帧只是 Colab CPU 上的参考。
- 不给 `TEST_SOURCE` 时比对用的是合成图，没有鼠。要看真实画面上的差，在第 2 格填一段视频。